# Module 08 — Practical GPU Benchmarking Laboratory

This notebook accompanies [the Module 08 lesson](../../docs/modules/08-gpu-benchmarking-lab.md). It measures VRAM availability, useful memory bandwidth, FP16/FP32 GEMM throughput, and small-kernel launch overhead.

The benchmark is designed for a Lightning AI NVIDIA T4. Results are workload measurements, not permanent properties of the GPU or guarantees of peak performance.

## Learning goals

By the end of this notebook, you should be able to:

- time asynchronous GPU work with CUDA events;
- distinguish physical VRAM from currently free VRAM;
- calculate useful-data bandwidth and achieved TFLOP/s;
- compare memory-bound, compute-oriented, and launch-sensitive work;
- save enough metadata to reproduce a measurement.

In [ ]:
from pathlib import Path
import sys

# Make the companion module importable from the repository root or this folder.
candidates = [Path.cwd() / 'code' / 'module-08', Path.cwd()]
module_dir = next(path for path in candidates if (path / 'benchmark_lab.py').exists())
sys.path.insert(0, str(module_dir))

import torch

from benchmark_lab import (
    bandwidth_experiment,
    gemm_experiment,
    launch_experiment,
    memory_experiment,
    print_device_report,
    print_roofline_interpretation,
    require_cuda,
    write_json,
)

# Set this to True on the Lightning T4 to run the GPU experiments.
RUN_GPU_EXPERIMENTS = False
WARMUP = 10
REPEATS = 30
DEVICE_NAME = 'cuda:0'

print('Companion module:', module_dir / 'benchmark_lab.py')
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## Timing protocol

Allocations and warmup belong outside the timed region. CUDA events measure elapsed time on the GPU stream; synchronization makes sure the result is ready before Python reads it.

![CUDA benchmark timing protocol](../../docs/assets/module-08/cuda-benchmark-timing-protocol.png)

The notebook defaults to `RUN_GPU_EXPERIMENTS = False` so that opening it does not unexpectedly allocate GPU memory. Set it to `True`, then execute the experiment cells on Lightning.

In [ ]:
if torch.cuda.is_available():
    device = require_cuda(DEVICE_NAME)
    metadata = print_device_report(device)
else:
    device = None
    metadata = {}
    print('CUDA is unavailable; run this cell on a Lightning GPU runtime.')

## Experiment A — VRAM capacity and availability

Physical memory, CUDA free memory, PyTorch allocated memory, and PyTorch reserved memory answer different questions. The safe planning rule is:

`usable memory <= physical VRAM - other allocations - runtime headroom`

The optional probe is disabled here because a successful allocation is not proof that a model, KV cache, and runtime will fit together.

In [ ]:
if RUN_GPU_EXPERIMENTS and device is not None:
    memory_experiment(device, probe_fraction=0.0)
else:
    print('Skipped. Set RUN_GPU_EXPERIMENTS = True on a CUDA runtime.')

## Experiment B — useful memory bandwidth

A copy accounts for one read and one write per element. A vector add accounts for two reads, one write, and one floating-point addition. The reported rate is useful bytes divided by device time; it is not automatically the physical memory-controller peak.

![Memory-bound vector add versus compute-oriented GEMM](../../docs/assets/module-08/memory-bound-vs-compute-oriented.png)

In [ ]:
bandwidth_results = []
if RUN_GPU_EXPERIMENTS and device is not None:
    bandwidth_results = bandwidth_experiment(
        device, vector_mib=128, warmup=WARMUP, repeats=REPEATS
    )
else:
    print('Skipped. Set RUN_GPU_EXPERIMENTS = True on a CUDA runtime.')

## Experiment C — FP16 and FP32 GEMM throughput

For `C = A @ B`, the approximate operation count is `2 * M * K * N`. Large shapes expose more parallel work and can amortize fixed launch costs; FP16 may use a high-throughput Tensor Core path for supported shapes.

In [ ]:
gemm_results = []
if RUN_GPU_EXPERIMENTS and device is not None:
    gemm_results = gemm_experiment(
        device, sizes=[512, 1024, 2048, 4096],
        warmup=WARMUP, repeats=REPEATS,
    )
else:
    print('Skipped. Set RUN_GPU_EXPERIMENTS = True on a CUDA runtime.')

In [ ]:
if gemm_results:
    import matplotlib.pyplot as plt

    for dtype_name in ['torch.float16', 'torch.float32']:
        rows = [result for result in gemm_results if result.label.startswith(dtype_name)]
        sizes = [int(result.label.split()[-1].split('x')[0]) for result in rows]
        tflops = [result.throughput for result in rows]
        plt.plot(sizes, tflops, marker='o', label=dtype_name)
    plt.xlabel('Square matrix dimension')
    plt.ylabel('Achieved TFLOP/s')
    plt.title('T4 GEMM throughput versus matrix size')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()
else:
    print('Run Experiment C first to plot GEMM throughput.')

## Experiment D — launch overhead and small work

This experiment repeats a small vector add. GPU event time estimates device execution; host-observed time also includes Python submission and synchronization. A small operation can be launch-sensitive even when the GPU has a high theoretical arithmetic peak.

In [ ]:
launch_results = []
if RUN_GPU_EXPERIMENTS and device is not None:
    launch_results = launch_experiment(
        device, elements=1024, launch_count=1000,
        warmup=WARMUP, repeats=REPEATS,
    )
else:
    print('Skipped. Set RUN_GPU_EXPERIMENTS = True on a CUDA runtime.')

## Interpret the measurements with the Roofline Model

The simplified model is `P <= min(P_peak, AI * BW)`. Low arithmetic intensity makes memory movement important; sufficiently high intensity makes the compute ceiling more relevant.

![Simplified Roofline Model](../../docs/assets/module-08/simplified-roofline-model.png)

In [ ]:
all_measurements = bandwidth_results + gemm_results + launch_results
if all_measurements:
    print_roofline_interpretation(
        all_measurements,
        memory_bandwidth_gbs=300.0,
        fp16_tflops=65.0,
        fp32_tflops=8.1,
    )
else:
    print('Run the experiments first to print the simplified ceilings.')

## Save a reproducible result

The companion script can save JSON directly with `--json`. The notebook can do the same after the experiment cells finish. The record includes device metadata, median latency, p10/p90 latency, and derived throughput.

In [ ]:
if metadata and all_measurements:
    write_json('module-08-results-from-notebook.json', metadata, all_measurements)
else:
    print('No CUDA measurements to save. Run the notebook on Lightning with RUN_GPU_EXPERIMENTS = True.')

## Interpretation checklist

Write down:

1. the GPU, PyTorch version, CUDA runtime, shapes, dtypes, warmup, and repetitions;
2. the observed free VRAM and whether the vector size changed the bandwidth result;
3. whether GEMM throughput rose with matrix size;
4. the difference between GPU event time and host-observed launch time;
5. one limitation that prevents this microbenchmark from being an end-to-end vLLM result.

A benchmark is evidence for a specific workload. Connect it to vLLM only after checking the real model shape, request mix, KV-cache traffic, scheduler behavior, and end-to-end latency.